# 🤖 Code Robot on Google Colab (GPU-Accelerated)

Welcome to **Code Robot** powered by **Next-Concept Modeling (NCM)** and the **Hybrid Consistency Architecture (HC-NCM)**!

This notebook lets you train, scale (up to 2B parameters), and run inference with free NVIDIA GPU acceleration (T4/V100/A100) on Google Colab.

### Step 1: Check GPU Acceleration
Make sure your Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> select `T4 GPU`).

Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

In [14]:
# %load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

,Registration State,Violation Description,count
0,CA,Bus Stop,50292
1,NJ,Double Parking,50168
2,NY,Fire Hydrant,50281
3,TX,No Parking,50053


In [21]:
import os

robot_path = '/content/robot'
if os.path.exists(robot_path):
    print(f"Files in {robot_path}:")
    for root, dirs, files in os.walk(robot_path):
        level = root.replace(robot_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{sub_indent}{f}")
else:
    print(f"{robot_path} directory does not exist.")

Files in /content/robot:
robot/
    run_hybrid_pipeline.py
    train_math.py
    run_self_training.py
    teach_math.py
    run_stage1.py
    run_pipeline.py
    .gitignore
    robot.py
    teach_robot.py
    README.md
    README_CUSTOM_NCM.md
    robot_colab.ipynb
    train_on_official_python.py
    custom_ncm/
        stage1_nursery.py
        generate.py
        lcm_flow.py
        ncm_model.py
        __init__.py
        official_python_harvester.py
        train_hybrid.py
        vqvae.py
        config.py
        code_tokenizer.py
        verifier.py
        train_vqvae.py
        teacher.py
        train_ncm.py
        curriculum.py
        hybrid_model.py
        dataset.py
        self_trainer.py
        task_bank.py
        __pycache__/
            generate.cpython-313.pyc
            train_hybrid.cpython-313.pyc
            __init__.cpython-313.pyc
            stage1_nursery.cpython-313.pyc
            official_python_harvester.cpython-313.pyc
            vqvae.cpython-313.p

In [3]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


### Step 2: Clone Code Robot Repository from GitHub

In [4]:
!rm -rf robot
!git clone https://github.com/nadeemmurtaza/robot.git
%cd robot

Cloning into 'robot'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 104 (delta 23), reused 94 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 882.13 KiB | 1.16 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/robot


### Step 3: Install Required Dependencies

In [2]:
!pip install -q torch torchvision torchaudio beautifulsoup4

In [1]:
!pip install -q black flake8 pylint ipywidgets

### Step 4: Inspect Model Parameter Tiers
View all available tiers: Small (~2.9M), Medium (~19.3M), Large (~108.3M), Option B (2.0B Frontier), and Hybrid (~3.6M).

In [5]:
!python robot.py --tiers

I0000 00:00:1793194766.702243    1806 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793194769.295829    1806 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793194769.335756    1806 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
  CODE ROBOT: MODEL PARAMETER TIERS & ARCHITECTURE

* Tier: Small (Baseline)
  - Total Parameters: 2,920,448 (~2.9M)
  - Concept VQ-VAE  : 974,336 params (2 enc / 2 dec layers)
  - NCM Generator   : 1,946,112 params (4 layers, 4 heads, dim=128)
  - Context Scales  : target=64 tokens, prefix=32 tokens
  - Levels (Hier.)  : [2, 8, 32, 64]

* Tier: Medium (Scaled)
  - Total Parameters: 19,334,144 (~19.3M)
  - Concept VQ-VAE  : 6,975,488 params (4 enc / 4 dec layers)
  

### Step 5: Run Stage 1 Nursery (Phonics & Syllables)
Watch the AI baby learn Python keywords, syllables, and bracket pairing.

In [6]:
!python run_stage1.py

I0000 00:00:1793194795.486620    2099 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793194796.381791    2099 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793194796.384492    2099 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
  STAGE 1: THE 'GOO-GOO GA-GA' NURSERY PHASE (AI BABY)
  Teaching the baby the alphabet, phonics, and atomic syllables of Python

--- PHASE 1: THE NEWBORN BABY (BEFORE TEACHING) ---
Observing what the untrained baby babbles when given basic phonics cues:

   Cue: de         -> Baby babbles: '??N?u,?_-????m?\x0c?7\x13G???x???\x18?y??'
   Cue: retu       -> Baby babbles: '??N?u,?_-????m?\x0c?7\x13G???x???\x18?y??'
   Cue: (          -> Baby babbles: '??N?u,?_-????m?\x

### Step 6: Train on the Official Python Standard Library & Docs
Trains on 1,415 verified functions harvested across 419 standard library modules.

In [7]:
!python train_on_official_python.py

I0000 00:00:1793194837.713565    2465 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793194838.598461    2465 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793194838.601368    2465 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
  TRAINING NCM MODEL ON OFFICIAL PYTHON STANDARD LIBRARY & DOCS
[*] Loaded 1415 official Python functions into training corpus.
[*] Training on device: cpu
[!] checkpoints/vqvae_pos.pt not found. Running VQ-VAE training first...
[*] Training VQ-VAE on device: cpu
[*] Total VQ-VAE Parameters: 974.3k
[*] Hierarchy Levels: [2, 8, 32, 64] (Context length: 64)
I0000 00:00:1793194838.974838    2465 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.116. The framework

### Step 7: Train & Benchmark the Hybrid Model (HC-NCM)
Combines 1-step Latent Consistency Mapping (LCM) + Diffusion Gemma Bidirectional Backbone + NCM Pyramid + Causal AR Syntax Realizer.

In [8]:
!python run_hybrid_pipeline.py

I0000 00:00:1793195057.693559    3544 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793195058.565614    3544 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793195058.568533    3544 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
  HYBRID CONSISTENCY NEXT-CONCEPT MODEL (HC-NCM) FOR CODE GENERATION
  Combining: LCM (1-Step Consistency) + Diffusion Gemma (Bidirectional Context)
             + NCM (Hierarchical Scaling) + Causal Gemma (Exact Syntax)

[*] Active Device: cpu

  ARCHITECTURAL BENCHMARK: SPEED, STEPS & FLOP REDUCTION

[Method]: Standard Autoregressive LLM (e.g. GPT/Gemma)
  - Forward Passes for 64 tokens : 64
  - Attention Mechanism          : Causal (Left-to-right only)
  - Genera

### Step 8: Interactive Code Generation
Generate code with parallel coarse-to-fine refinement on any prompt:

In [11]:
!python robot.py --hybrid --prompt  "Return the square of x\ndef square(x):\n"

I0000 00:00:1793195377.804086    5033 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793195378.687088    5033 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793195378.689826    5033 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
  WELCOME TO CODE ROBOT (NEXT-CONCEPT & HYBRID CONSISTENCY MODELING)
[*] Running in HYBRID CONSISTENCY MODE (1-Step LCM + Bidirectional + NCM + Causal AR)

[+] Running Hybrid Generation on prompt:
Return the square of x\ndef square(x):\n
[*] Chosen Source: NCM Hierarchy (Fallback)
[*] Generated Code:
he square of x\ndef square(x):\n
        ifif..sseereeme::                                       
[*] AST Valid: False


In [12]:
!python robot.py --prompt  "Filter even numbers from list\ndef filter_evens(nums):\n"

I0000 00:00:1793195440.576121    5317 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793195441.452160    5317 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793195441.455130    5317 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
  WELCOME TO CODE ROBOT (NEXT-CONCEPT & HYBRID CONSISTENCY MODELING)
[*] Loading Code Robot from: checkpoints/generator_official_python.pt

[+] Running generation on prompt:
Filter even numbers from list\ndef filter_evens(nums):\n

  NEXT-CONCEPT MODELING (NCM) COARSE-TO-FINE CODE GENERATION
[*] Prompt (Prefix):
Filter even numbers from list\ndef filter_evens(nums):\n
----------------------------------------------------------------------

[Level 0] (Scale: 2 paralle

In [30]:
%%writefile teach_math.py
# Creating an algebra-focused training dataset for Code Robot
math_corpus = [
    "def solve_linear(a, b):\n    # Solve a*x + b = 0\n    if a == 0:\n        return None\n    return -b / a\n",
    "def quadratic_formula(a, b, c):\n    # Calculate discriminant\n    d = b**2 - 4*a*c\n    if d < 0:\n        return None\n    x1 = (-b + d**0.5) / (2*a)\n    x2 = (-b - d**0.5) / (2*a)\n    return x1, x2\n",
    "def get_slope(x1, y1, x2, y2):\n    # Find slope m of line\n    if x1 == x2:\n        return float('inf')\n    return (y2 - y1) / (x2 - x1)\n",
    "def evaluate_polynomial(coeffs, x):\n    # Evaluate custom polynomial\n    return sum(c * (x ** i) for i, c in enumerate(coeffs))\n",
    "def is_perfect_square(n):\n    # Check if algebraic root is integer\n    if n < 0:\n        return False\n    root = int(n**0.5)\n    return root * root == n\n"
] * 100

print(f"Successfully created an algebraic dataset with {len(math_corpus)} records in /content/robot/teach_math.py")

Overwriting teach_math.py


In [37]:
%%writefile train_math.py
import sys
import os
# Ensure the robot source directory and its custom subfolders are in Python's search path
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('./custom_ncm'))
sys.path.append(os.path.abspath('./ncm'))

import torch
from teach_math import math_corpus
from ncm_model import NextConceptGenerator
from code_tokenizer import HFCodeTokenizer
from vqvae import CodeVQVAE

print(f"Training Code Robot on {len(math_corpus)} algebraic functions...")
tokenizer = HFCodeTokenizer()

# Initialize and load the trained VQ-VAE model
vqvae = CodeVQVAE(vocab_size=tokenizer.vocab_size)
try:
    vqvae_state = torch.load('checkpoints/vqvae_pos.pt', map_location='cpu')
    vqvae.load_state_dict(vqvae_state, strict=False)
    print("Loaded VQ-VAE baseline encoder/decoder successfully!")
except Exception as e:
    print("Initializing VQ-VAE from scratch (checkpoints/vqvae_pos.pt not found or mismatch).")

# Initialize the Next-Concept Generator with the VQ-VAE
model = NextConceptGenerator(vqvae=vqvae)
prefix_len = 32 # Default prefix length expected by the model

# Use nursery stage1 weights if available as a baseline
try:
    checkpoint = torch.load('checkpoints/nursery/stage1_baby.pt', map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    print("Loaded baseline weights successfully!")
except Exception as e:
    print("Starting training from scratch.")

# Simple fine-tuning pass simulation for demonstration
device = 'cpu'
model.to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print("Training initialized. Starting algebra ingestion...")
for epoch in range(1, 4):
    total_loss = 0.0
    for func in math_corpus[:20]: # train on subset for speed
        tokens = tokenizer.encode(func)

        # Standardize prefix sequence 'x' to match prefix_len (32 tokens)
        if len(tokens) < prefix_len + 1:
            # Pad with 0s if short
            tokens = tokens + [0] * (prefix_len + 1 - len(tokens))

        # Slice to ensure prefix_ids are exactly prefix_len (32)
        prefix_tokens = tokens[:prefix_len]
        target_tokens = tokens[1:prefix_len+1] # target sequence matching the offset

        x = torch.tensor(prefix_tokens).unsqueeze(0).to(device)
        y = torch.tensor(target_tokens).unsqueeze(0).to(device)

        optimizer.zero_grad()
        # Forward pass yields: logits_list, targets_list, and loss
        _, _, loss = model(x, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/3 | Algebra Training Loss: {total_loss/20:.4f}")

torch.save({'model_state_dict': model.state_dict()}, 'checkpoints/generator_math.pt')
print("Saved mathematical concept checkpoint to checkpoints/generator_math.pt!")

Overwriting train_math.py


In [38]:
%cd /content/robot
!python train_math.py

/content/robot
I0000 00:00:1793196361.604924    9399 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793196362.491504    9399 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793196362.494488    9399 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
Successfully created an algebraic dataset with 500 records in /content/robot/teach_math.py
Training Code Robot on 500 algebraic functions...
Initializing VQ-VAE from scratch (checkpoints/vqvae_pos.pt not found or mismatch).
Starting training from scratch.
Training initialized. Starting algebra ingestion...
I0000 00:00:1793196365.706775    9399 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.116. The framework PJRT API version is 0.116.
I0000 0

In [39]:
import sys
import os
# Add the robot folders to the path so we can import the models
sys.path.append('/content/robot')
sys.path.append('/content/robot/custom_ncm')

import torch
from code_tokenizer import HFCodeTokenizer
from vqvae import CodeVQVAE
from ncm_model import NextConceptGenerator
from generate import generate_code_ncm

# Initialize tokenizer and baseline structures
tokenizer = HFCodeTokenizer()
vqvae = CodeVQVAE(vocab_size=tokenizer.vocab_size)

# Load the trained VQ-VAE model
try:
    vqvae_state = torch.load('/content/robot/checkpoints/vqvae_pos.pt', map_location='cpu')
    vqvae.load_state_dict(vqvae_state, strict=False)
except Exception as e:
    print("VQ-VAE loaded with default configuration.")

# Load Next-Concept Generator with math checkpoint
model = NextConceptGenerator(vqvae=vqvae)
try:
    checkpoint = torch.load('/content/robot/checkpoints/generator_math.pt', map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    print("\n[Success] Loaded trained math checkpoint successfully!")
except Exception as e:
    print("[Fallback] Loaded from scratch or default configuration.")

# Prompt the model to write code for a matrix multiplication transpose operation
prompt = "def matrix_transpose(matrix):\n    # Teach matrix transpose\n"
print(f"\nEvaluating trained robot with algebraic/matrix prompt:\n{prompt}")

# Evaluate generation
model.eval()
with torch.no_grad():
    tokens = tokenizer.encode(prompt)
    # Ensure sequence length matches the expected prefix size (32)
    if len(tokens) < 32:
        tokens = tokens + [0] * (32 - len(tokens))
    else:
        tokens = tokens[:32]

    x = torch.tensor(tokens).unsqueeze(0)
    # Run forward step to generate top level tokens
    logits_list, _, _ = model(x, x) # using input sequence dummy target
    # Decode the top-level logits to inspect structural generation
    last_logits = logits_list[-1][0] # final level logits
    predicted_ids = torch.argmax(last_logits, dim=-1).tolist()
    decoded_out = tokenizer.decode(predicted_ids)

    print("\nRobot Model structural generation output:\n", decoded_out)

ImportError: cannot import name 'generate_code_ncm' from 'generate' (/content/robot/custom_ncm/generate.py)

In [26]:
import inspect
import sys
import os
sys.path.append('/content/robot')
sys.path.append('/content/robot/custom_ncm')
from ncm_model import NextConceptGenerator

# Print constructor signature
print("NextConceptGenerator __init__ signature:")
print(inspect.signature(NextConceptGenerator.__init__))

# Search train_on_official_python.py to see how it instantiates the generator
with open('/content/robot/train_on_official_python.py', 'r') as f:
    for line in f:
        if 'NextConceptGenerator' in line:
            print("Instantiation in train_on_official_python.py:", line.strip())

NextConceptGenerator __init__ signature:
(self, vqvae: custom_ncm.vqvae.CodeVQVAE, prefix_len: int = 32, d_model: int = 128, n_head: int = 4, n_layer: int = 4, dropout: float = 0.1, codebook_size: int = 256)
Instantiation in train_on_official_python.py: from custom_ncm.ncm_model import NextConceptGenerator
Instantiation in train_on_official_python.py: generator = NextConceptGenerator(


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


In [27]:
import inspect
import sys
import os
sys.path.append('/content/robot')
sys.path.append('/content/robot/custom_ncm')
from ncm_model import NextConceptGenerator

# Print constructor signature
print("NextConceptGenerator __init__ signature:")
print(inspect.signature(NextConceptGenerator.__init__))

# Search train_on_official_python.py to see how it instantiates the generator
with open('/content/robot/train_on_official_python.py', 'r') as f:
    for line in f:
        if 'NextConceptGenerator' in line:
            print("Instantiation in train_on_official_python.py:", line.strip())

NextConceptGenerator __init__ signature:
(self, vqvae: custom_ncm.vqvae.CodeVQVAE, prefix_len: int = 32, d_model: int = 128, n_head: int = 4, n_layer: int = 4, dropout: float = 0.1, codebook_size: int = 256)
Instantiation in train_on_official_python.py: from custom_ncm.ncm_model import NextConceptGenerator
Instantiation in train_on_official_python.py: generator = NextConceptGenerator(


In [35]:
import inspect
import sys
import os
sys.path.append('/content/robot')
sys.path.append('/content/robot/custom_ncm')
from ncm_model import NextConceptGenerator

# Let's inspect the return signature/statements of the forward method
source_code = inspect.getsource(NextConceptGenerator.forward)
print("Source of NextConceptGenerator.forward:")
print('\n'.join(source_code.split('\n')[:35]))

Source of NextConceptGenerator.forward:
    def forward(
        self,
        prefix_ids: torch.Tensor,
        target_ids: torch.Tensor
    ) -> Tuple[List[torch.Tensor], List[torch.Tensor], torch.Tensor]:
        """
        Training forward pass.
        Given prefix_ids and target_ids, extracts target concept codes from frozen VQ-VAE,
        feeds the hierarchical representation through the transformer with block-causal mask,
        and computes cross-entropy loss against the true concept codes at each level.
        """
        B = prefix_ids.shape[0]
        device = prefix_ids.device
        
        # 1. Extract ground-truth concept codes and intermediate representations from VQ-VAE
        with torch.no_grad():
            B_tgt, L_tgt = target_ids.shape
            pos = torch.arange(L_tgt, device=device).unsqueeze(0)
            h_tgt = self.vqvae.token_emb(target_ids) + self.vqvae.pos_emb(pos)
            for block in self.vqvae.enc_blocks:
                h_tgt = block(

In [36]:
import inspect
import sys
import os
sys.path.append('/content/robot')
sys.path.append('/content/robot/custom_ncm')
from ncm_model import NextConceptGenerator

# Let's inspect the return signature/statements of the forward method
source_code = inspect.getsource(NextConceptGenerator.forward)
print("Source of NextConceptGenerator.forward:")
print('\n'.join(source_code.split('\n')[:35])) # Print the first 35 lines to check the returns

Source of NextConceptGenerator.forward:
    def forward(
        self,
        prefix_ids: torch.Tensor,
        target_ids: torch.Tensor
    ) -> Tuple[List[torch.Tensor], List[torch.Tensor], torch.Tensor]:
        """
        Training forward pass.
        Given prefix_ids and target_ids, extracts target concept codes from frozen VQ-VAE,
        feeds the hierarchical representation through the transformer with block-causal mask,
        and computes cross-entropy loss against the true concept codes at each level.
        """
        B = prefix_ids.shape[0]
        device = prefix_ids.device
        
        # 1. Extract ground-truth concept codes and intermediate representations from VQ-VAE
        with torch.no_grad():
            B_tgt, L_tgt = target_ids.shape
            pos = torch.arange(L_tgt, device=device).unsqueeze(0)
            h_tgt = self.vqvae.token_emb(target_ids) + self.vqvae.pos_emb(pos)
            for block in self.vqvae.enc_blocks:
                h_tgt = block(

In [34]:
%cd /content/robot
!python train_math.py

/content/robot
I0000 00:00:1793196291.103630    9035 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793196291.979364    9035 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793196291.982301    9035 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
Successfully created an algebraic dataset with 500 records in /content/robot/teach_math.py
Training Code Robot on 500 algebraic functions...
Initializing VQ-VAE from scratch (checkpoints/vqvae_pos.pt not found or mismatch).
Starting training from scratch.
Training initialized. Starting algebra ingestion...
Traceback (most recent call last):
  File "/content/robot/train_math.py", line 64, in <module>
    logits, _ = model(x, y)
    ^^^^^^^^^
ValueError

### Step 9 (Optional): Mount Google Drive to Save Checkpoints
Persist your trained models into your Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/robot_checkpoints
# !cp -r checkpoints/* /content/drive/MyDrive/robot_checkpoints/
# print("Checkpoints successfully saved to Google Drive!")